In [ ]:
import pandas as pd
import duckdb
from matplotlib import pyplot as plt
import numpy as np
from IPython.display import display

In [ ]:
path = (r'\Users\DELL\OneDrive\Desktop\docs\duck\\')
aisles=pd.read_csv(path+"aisles.csv")
departments=pd.read_csv(path+"departments.csv")
orders=pd.read_csv(path+"orders.csv")
products=pd.read_csv(path+"products.csv")
order_products__train=pd.read_csv(path+"order_products__train.csv")
order_products__prior=pd.read_csv(path+"order_products__prior.csv")

  # 3. Display DataFrames
dfs = [aisles, departments, order_products__train,order_products__prior,orders, products]
for df in dfs:
    display(df.head())


,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
3,4,instant foods
4,5,marinades meat preparation


,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol


,order_id,product_id,add_to_cart_order,reordered
0,1,49302,1,1
1,1,11109,2,1
2,1,10246,3,0
3,1,49683,4,0
4,1,43633,5,1


,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0


,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


In [ ]:
# 4. Register with DuckDB
# create dictionary to map  table names to  DataFrames
register_dict = {
    "aisles": aisles,
    "departments": departments,
    "orders": orders,
    "products": products,
    "order_products__train": order_products__train,
    "order_products__prior": order_products__prior
}

for name, df in register_dict.items():
    duckdb.register(name, df)

Join two tables together:

SELECT * FROM tbl JOIN other_table

ON tbl.key = other_table.key;

#Select a 10% sample from a table:

SELECT * FROM tbl


TABLESAMPLE 10%;

#Select a sample of 10 rows from a table:

SELECT * FROM tbl

TABLESAMPLE 10 ROWS;

question 6

Join order_products and products
to find the most frequently
ordered product.

In [ ]:
duckdb.sql("""
SELECT 
    products.product_name,
    COUNT(combined_order_products_tables.product_id) AS times_ordered
FROM (
    SELECT * FROM order_products__prior
    UNION ALL
    SELECT * FROM order_products__train
) AS combined_order_products_tables
JOIN products
  ON combined_order_products_tables.product_id = products.product_id
GROUP BY products.product_name
ORDER BY times_ordered DESC
LIMIT 3;
""").show()

┌────────────────────────┬───────────────┐
│      product_name      │ times_ordered │
│        varchar         │     int64     │
├────────────────────────┼───────────────┤
│ Banana                 │        491291 │
│ Bag of Organic Bananas │        394930 │
│ Organic Strawberries   │        275577 │
└────────────────────────┴───────────────┘



question 6

Join order_products and products
 most frequently ordered product.
     is BANANA ORDERED 491,291 TIMES
2nd most ordered is
Bag of Organic Bananaas 394 930

Question 1
Join Products and aisles to show 
Product names and their aisle names


#
Join two tables together:

SELECT * FROM tbl JOIN other_table

ON tbl.key = other_table.key;

#Select a 10% sample from a table:

SELECT * FROM tbl

TABLESAMPLE 10%;

In [ ]:
duckdb.sql("""
SELECT 
     products.product_name,
     aisles.aisle,
     aisles.aisle_id
FROM products
JOIN aisles
  ON products.product_id = aisles.aisle_id
""").show()

┌───────────────────────────────────────────────────────────────────┬────────────────────────────┬──────────┐
│                           product_name                            │           aisle            │ aisle_id │
│                              varchar                              │          varchar           │  int64   │
├───────────────────────────────────────────────────────────────────┼────────────────────────────┼──────────┤
│ Chocolate Sandwich Cookies                                        │ prepared soups salads      │        1 │
│ All-Seasons Salt                                                  │ specialty cheeses          │        2 │
│ Robust Golden Unsweetened Oolong Tea                              │ energy granola bars        │        3 │
│ Smart Ones Classic Favorites Mini Rigatoni With Vodka Cream Sauce │ instant foods              │        4 │
│ Green Chile Anytime Sauce                                         │ marinades meat preparation │        5 │
│ Dry Nose

# i used
SELECT 
     products.product_name,
     aisles.aisle,
     aisles.aisle_id
because when I initially type 
SELECT 
     products.product_name,
     #aisles.aisle_name
 ERROR
was aisles table has no aisle_name column

question 7
#Find the average
order_hour_of_day for 
#each day of the week

In [ ]:
duckdb.sql("""
SELECT 
    order_dow,
    order_hour_of_day,
    AVG(order_hour_of_day) 
FROM 
    orders
GROUP BY 
    order_dow,
    order_hour_of_day
ORDER BY 
    order_dow
""").show()

┌───────────┬───────────────────┬────────────────────────┐
│ order_dow │ order_hour_of_day │ avg(order_hour_of_day) │
│   int64   │       int64       │         double         │
├───────────┼───────────────────┼────────────────────────┤
│         0 │                 3 │                    3.0 │
│         0 │                 2 │                    2.0 │
│         0 │                18 │                   18.0 │
│         0 │                12 │                   12.0 │
│         0 │                20 │                   20.0 │
│         0 │                 6 │                    6.0 │
│         0 │                 5 │                    5.0 │
│         0 │                 4 │                    4.0 │
│         0 │                16 │                   16.0 │
│         0 │                19 │                   19.0 │
│         · │                 · │                     ·  │
│         · │                 · │                     ·  │
│         · │                 · │                     · 

#4
Find average number of items per order

In [ ]:
duckdb.sql("""
SELECT 
    AVG(items_per_order) AS avg_items_per_order
FROM (
    SELECT 
        orders.order_id,
        COUNT(op.product_id) AS items_per_order
    FROM orders 
    JOIN order_products__train AS op
      ON orders.order_id = op.order_id
    GROUP BY orders.order_id
) 
""").show()


┌─────────────────────┐
│ avg_items_per_order │
│       double        │
├─────────────────────┤
│  10.552759338155157 │
└─────────────────────┘



#3
List the names of products in the "alcohol" department

In [ ]:
duckdb.sql("""
SELECT 
    products.product_name
FROM products 
JOIN departments
  ON products.department_id = departments.department_id
WHERE departments.department = 'Alcohol'
ORDER BY products.product_name;
""").show()


┌──────────────┐
│ product_name │
│   varchar    │
└──────────────┘
     0 rows   



In [ ]:
duckdb.sql("""
SELECT DISTINCT 
    products.product_name
FROM order_products__prior AS op
JOIN products 
  ON op.product_id = products.product_id
WHERE op.reordered = 0
ORDER BY products.product_name;
""").show()


┌────────────────────────────────────────────────┐
│                  product_name                  │
│                    varchar                     │
├────────────────────────────────────────────────┤
│ #2 Coffee Filters                              │
│ #2 Cone White Coffee Filters                   │
│ #2 Mechanical Pencils                          │
│ #4 Natural Brown Coffee Filters                │
│ & Go! Hazelnut Spread + Pretzel Sticks         │
│ 'Swingtop' Premium Lager                       │
│ (70% Juice!) Mountain Raspberry Juice Squeeze  │
│ +Energy Black Cherry Vegetable & Fruit Juice   │
│ .5\" Waterproof Tape                           │
│ 0 Calorie Acai Raspberry Water Beverage        │
│      ·                                         │
│      ·                                         │
│      ·                                         │
│ Coffee Cake                                    │
│ Coffee Cake Bites                              │
│ Coffee Cake, Cinnamon Walnut 

#5
List users who have placed morethan 5 orders

https://duckdb.org/docs/current/sql/query_syntax/having?utm_source=copilot.com

Count the number of entries in the addresses table that belong to each different city, filtering out cities with a count below 50:

SELECT city, count(*)
FROM addresses
GROUP BY city
HAVING count(*) >= 50;

In [ ]:
duckdb.sql("""
SELECT 
    user_id,
    COUNT(order_id) AS total_orders
FROM orders
GROUP BY user_id
HAVING COUNT(order_id) > 5
ORDER BY total_orders ASC
""").show()

┌─────────┬──────────────┐
│ user_id │ total_orders │
│  int64  │    int64     │
├─────────┼──────────────┤
│   61826 │            6 │
│   62627 │            6 │
│   62750 │            6 │
│   62625 │            6 │
│   63334 │            6 │
│   62749 │            6 │
│   63343 │            6 │
│   61864 │            6 │
│   63071 │            6 │
│   63730 │            6 │
└─────────┴──────────────┘
  10 rows      2 columns

